#**Brain Tumor Classification Pipeline: From Imagery to Insights**
## **Digital Egypt Pioneers Initiative (DEPI) - Microsoft AI & Data Science Track**
### **Capstone Project**

---

### **Project Overview**
This project is developed as part of the **DEPI Microsoft Machine Learning Engineer Track**. It demonstrates an end-to-end cloud-based ML pipeline for classifying brain tumors (**Glioma, Meningioma, Pituitary, No Tumor**) using the **BRISC 2025 Dataset**.

**Methodology:**
We leverage **Computer Vision** techniques to extract geometric and texture features from MRI segmentation masks, transforming raw imagery into a structured tabular dataset. This data is then processed using **Microsoft Azure Automated ML** to build, evaluate, and deploy a high-performance classification model, showcasing the power of Azure in medical diagnostics.

---

### **Project Team (DEPI Trainees)**
**Microsoft AI & Data Science Track**

1.  **Abdelrahman Hisham Ismail** - *(Team Leader)*
2.  **Abdelrahman Mahmoud Ahmed**
3.  **Omar Tarek Emam**
4.  **Abdallah Mohamed Fahmy**
5.  **Amgad Mohammed Mohammed**
6.  **Zyad Atef**

---

###**Tech Stack & Tools**
* **Platform:** Microsoft Azure Machine Learning.
* **Data Source:** BRISC 2025 (Nature Scientific Data).
* **Preprocessing:** Python, OpenCV, Pandas (Google Colab).
* **Model Training:** Azure Automated ML (Classification).
* **Deployment:** Azure Real-time Endpoint.

---

In [ ]:
# =============================================================================
# Setup & Configuration (Google Colab)
# =============================================================================
# Mount Google Drive and define all dataset paths.
# The dataset follows the BRISC 2025 structure with separate
# classification and segmentation directories.
# =============================================================================

import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/DEPI - GP')
DATASET_PATH = PROJECT_ROOT / 'brisc2025'
SEG_PATH     = DATASET_PATH / 'segmentation_task'
CLF_PATH     = DATASET_PATH / 'classification_task'

print('Paths set ✅')

## Exploratory Data Analysis (EDA)

**Goals:**
1. Count images per class and verify train/test split balance.
2. Visualize sample MRI slices alongside their segmentation masks.
3. Check image dimensions for consistency.

In [ ]:
# Install required dependencies (Colab only)
!pip install -q opencv-python-headless

In [ ]:
# =============================================================================
# 1. Class Distribution
# =============================================================================
# Count images per class across train/test splits to verify
# that the dataset is balanced and complete.
# Expected: ~5,000 train / ~1,000 test across 4 classes.
# =============================================================================

import matplotlib.pyplot as plt
import numpy as np
import cv2

CLASSES = ['glioma', 'meningioma', 'pituitary', 'no_tumor']

splits = {}
for split in ['train', 'test']:
    counts = {}
    for cls in CLASSES:
        folder = CLF_PATH / split / cls
        counts[cls] = len(list(folder.glob('*.jpg')))
    splits[split] = counts

# Grouped bar chart
x = np.arange(len(CLASSES))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, [splits['train'][c] for c in CLASSES], w, label='Train')
ax.bar(x + w/2, [splits['test'][c]  for c in CLASSES], w, label='Test')
ax.set_xticks(x)
ax.set_xticklabels(CLASSES)
ax.set_ylabel('Count')
ax.set_title('Images per Class')
ax.legend()
plt.tight_layout()
plt.show()

print('Train:', splits['train'])
print('Test :', splits['test'])

In [ ]:
# =============================================================================
# 2. Sample Visualization — MRI Slices vs. Segmentation Masks
# =============================================================================
# Display one sample per tumor class (glioma, meningioma, pituitary)
# with its corresponding segmentation mask.
# Note: 'no_tumor' is excluded — no tumor region to segment.
# =============================================================================

SEG_CLASSES = ['glioma', 'meningioma', 'pituitary']
CODE_MAP = {'glioma': '_gl_', 'meningioma': '_me_', 'pituitary': '_pi_'}

fig, axes = plt.subplots(2, 3, figsize=(12, 6))

for i, cls in enumerate(SEG_CLASSES):
    img_dir  = SEG_PATH / 'train' / 'images'
    mask_dir = SEG_PATH / 'train' / 'masks'

    # Select one sample by matching the tumor code in the filename
    code   = CODE_MAP[cls]
    sample = next(f for f in sorted(img_dir.glob('*.jpg')) if code in f.name)

    img  = cv2.cvtColor(cv2.imread(str(sample)), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(mask_dir / sample.with_suffix('.png').name),
                      cv2.IMREAD_GRAYSCALE)

    axes[0, i].imshow(img)
    axes[0, i].set_title(cls)
    axes[0, i].axis('off')

    axes[1, i].imshow(mask, cmap='gray')
    axes[1, i].set_title('Mask')
    axes[1, i].axis('off')

fig.suptitle('MRI Slice vs Segmentation Mask', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 3. Image Dimensions Consistency Check
# =============================================================================
# Verify whether all images share the same resolution.
# Inconsistent sizes may require resizing during preprocessing.
# =============================================================================

from collections import Counter

sizes = []
for img_path in (SEG_PATH / 'train' / 'images').glob('*.jpg'):
    h, w = cv2.imread(str(img_path)).shape[:2]
    sizes.append((h, w))

size_counts = Counter(sizes)
print(f'Total train images: {len(sizes)}')
print(f'Unique sizes: {len(size_counts)}')
for size, count in size_counts.most_common(5):
    print(f'  {size[1]}x{size[0]} — {count} images')

### 1.1  Class-Distribution Pie Charts
Visualize the **proportion** of each class within the train and test
splits using pie charts — complements the bar chart above.

In [ ]:
# =============================================================================
# 1.1 Pie Charts — Train & Test Class Proportions
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, split in zip(axes, ['train', 'test']):
    labels = list(splits[split].keys())
    sizes  = list(splits[split].values())
    colors = plt.cm.Set2.colors[:len(labels)]
    wedges, texts, autotexts = ax.pie(
        sizes, labels=labels, autopct='%1.1f%%',
        colors=colors, startangle=140, textprops={'fontsize': 10})
    ax.set_title(f'{split.capitalize()} Split', fontsize=13)

fig.suptitle('Class Proportions — Train vs Test', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### 1.2  Multi-Sample Grid
Display a **4 × 3 grid** of random MRI samples (4 per class) to give a
broader view of the dataset variety.

In [ ]:
# =============================================================================
# 1.2 Multi-Sample MRI Grid (4 per class)
# =============================================================================
import random

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for row, cls in enumerate(SEG_CLASSES):
    code = CODE_MAP[cls]
    img_dir = SEG_PATH / 'train' / 'images'
    candidates = sorted([f for f in img_dir.glob('*.jpg') if code in f.name])
    samples = random.sample(candidates, min(4, len(candidates)))
    for col, sample in enumerate(samples):
        img = cv2.cvtColor(cv2.imread(str(sample)), cv2.COLOR_BGR2RGB)
        axes[row, col].imshow(img)
        axes[row, col].set_title(f'{cls}', fontsize=10)
        axes[row, col].axis('off')

fig.suptitle('Random MRI Samples — Segmentation Classes', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

### 1.3  Image Dimension Distribution
A **bar chart** of the top-10 image resolutions found in the training
segmentation images — clearer than the text listing above.

In [ ]:
# =============================================================================
# 1.3 Image Dimension Distribution (bar chart)
# =============================================================================

dim_labels = [f'{w}×{h}' for (h, w), _ in size_counts.most_common(10)]
dim_counts = [c for _, c in size_counts.most_common(10)]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(dim_labels[::-1], dim_counts[::-1], color='steelblue')
ax.set_xlabel('Number of Images')
ax.set_title('Top-10 Image Resolutions (Train — Segmentation)')
for bar in bars:
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            str(int(bar.get_width())), va='center', fontsize=9)
plt.tight_layout()
plt.show()

### 1.4  MRI + Mask Overlay
Overlay the segmentation mask on top of the original MRI image with a
semi-transparent colormap.  This clearly shows what the masks capture.

In [ ]:
# =============================================================================
# 1.4 MRI + Mask Overlay Visualization
# =============================================================================

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for i, cls in enumerate(SEG_CLASSES):
    code   = CODE_MAP[cls]
    img_dir  = SEG_PATH / 'train' / 'images'
    mask_dir = SEG_PATH / 'train' / 'masks'
    sample = next(f for f in sorted(img_dir.glob('*.jpg')) if code in f.name)

    img  = cv2.cvtColor(cv2.imread(str(sample)), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(mask_dir / sample.with_suffix('.png').name),
                      cv2.IMREAD_GRAYSCALE)

    axes[0, i].imshow(img)
    axes[0, i].set_title(f'{cls} — MRI', fontsize=11)
    axes[0, i].axis('off')

    axes[1, i].imshow(img)
    axes[1, i].imshow(mask, cmap='Reds', alpha=0.45)
    axes[1, i].set_title(f'{cls} — Mask Overlay', fontsize=11)
    axes[1, i].axis('off')

fig.suptitle('MRI Slice with Segmentation Mask Overlay', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## Feature Extraction

**Goal:** Extract geometric and texture features from segmentation masks to build a structured tabular dataset for classification.

**Features:**
- **Geometric:** area, perimeter, solidity, eccentricity, bounding box aspect ratio.
- **Texture (GLCM):** contrast, correlation, energy, homogeneity.
- **Metadata:** anatomical view (axial / coronal / sagittal), tumor label.

In [ ]:
# Install additional dependencies (Colab only)
!pip install -q scikit-image

In [ ]:
# =============================================================================
# 4. Feature Extraction Pipeline
# =============================================================================
# For each MRI image that has a segmentation mask, extract:
#   - Geometric features  : area, perimeter, solidity, eccentricity, bbox ratio
#   - Texture features    : GLCM contrast, correlation, energy, homogeneity
#   - Metadata            : anatomical view and tumor label (from filename)
#
# The output is a pandas DataFrame — one row per image.
# =============================================================================

import pandas as pd
from skimage.feature import graycomatrix, graycoprops
from skimage.measure import regionprops, label as sk_label

# --- Filename parser ---
TUMOR_MAP = {'gl': 'glioma', 'me': 'meningioma', 'pi': 'pituitary', 'nt': 'no_tumor'}
VIEW_MAP  = {'ax': 'axial',  'co': 'coronal',    'sa': 'sagittal'}

def parse_filename(name):
    """Extract split, label, and view from BRISC filename."""
    parts = name.stem.split('_')   # e.g. brisc2025_train_00001_gl_ax_t1
    return {
        'split': parts[1],
        'label': TUMOR_MAP[parts[3]],
        'view' : VIEW_MAP[parts[4]],
    }

# --- Geometric features ---
def geometric_features(mask):
    """Compute shape-based features from a binary mask."""
    labeled = sk_label(mask > 0)
    regions = regionprops(labeled)
    if not regions:
        return {k: 0.0 for k in ['area', 'perimeter', 'solidity',
                                  'eccentricity', 'bbox_ratio']}
    r = max(regions, key=lambda x: x.area)   # largest connected component
    h = r.bbox[2] - r.bbox[0]
    w = r.bbox[3] - r.bbox[1]
    return {
        'area'        : r.area,
        'perimeter'   : r.perimeter,
        'solidity'    : r.solidity,
        'eccentricity': r.eccentricity,
        'bbox_ratio'  : h / w if w > 0 else 0.0,
    }

# --- Texture features (GLCM) ---
def texture_features(gray_img, mask):
    """Compute GLCM texture features within the masked region."""
    roi = cv2.bitwise_and(gray_img, gray_img, mask=(mask > 0).astype(np.uint8))
    glcm = graycomatrix(roi, distances=[1], angles=[0], levels=256,
                        symmetric=True, normed=True)
    return {
        'contrast'   : graycoprops(glcm, 'contrast')[0, 0],
        'correlation': graycoprops(glcm, 'correlation')[0, 0],
        'energy'     : graycoprops(glcm, 'energy')[0, 0],
        'homogeneity': graycoprops(glcm, 'homogeneity')[0, 0],
    }

print('Feature functions defined ✅')

In [ ]:
# =============================================================================
# 5. Run Feature Extraction (Train + Test)
# =============================================================================
# Loop over all images that have segmentation masks and build
# the final tabular dataset.
# =============================================================================

def extract_dataset_features(splits=['train', 'test']):
    records = []
    
    for split in splits:
        img_dir  = SEG_PATH / split / 'images'
        mask_dir = SEG_PATH / split / 'masks'

        img_files = sorted(img_dir.glob('*.jpg'))
        print(f'Processing {split}: {len(img_files)} images ...')

        for img_path in img_files:
            mask_path = mask_dir / img_path.with_suffix('.png').name
            if not mask_path.exists():
                continue

            # Read image and mask
            gray = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

            # Parse filename metadata
            meta = parse_filename(img_path)

            # Extract features
            geo  = geometric_features(mask)
            tex  = texture_features(gray, mask)

            records.append({**meta, **geo, **tex, 'filename': img_path.name})
            
    return pd.DataFrame(records)

df = extract_dataset_features(['train', 'test'])
print(f'\nDataset shape: {df.shape}')
df.head()

In [ ]:
# =============================================================================
# 6. Save Tabular Dataset
# =============================================================================
# Export the extracted features as a CSV file.
# This file will be used for Azure AutoML classification.
# =============================================================================

output_path = DATASET_PATH / 'features.csv'
df.to_csv(output_path, index=False)

print(f'Saved to: {output_path}')
print(f'Shape   : {df.shape}')
print(f'Columns : {list(df.columns)}')
print(f'\nLabel distribution:')
print(df['label'].value_counts())

## Feature Analysis & Visualization

Now that the features are extracted, let's **visualize** them to
understand the data distribution, correlations, and class separability.

### 2.1  Feature Distributions (Histograms + KDE)
Show the distribution of each numeric feature, colored by tumor label.

In [ ]:
# =============================================================================
# 2.1 Feature Distributions — Histograms with KDE
# =============================================================================
import seaborn as sns

FEATURES = ['area', 'perimeter', 'solidity', 'eccentricity',
            'bbox_ratio', 'contrast', 'correlation', 'energy', 'homogeneity']

fig, axes = plt.subplots(3, 3, figsize=(18, 12))

for ax, feat in zip(axes.flat, FEATURES):
    for label in df['label'].unique():
        subset = df[df['label'] == label]
        ax.hist(subset[feat], bins=40, alpha=0.5, label=label, density=True)
    ax.set_title(feat, fontsize=12)
    ax.legend(fontsize=8)

fig.suptitle('Feature Distributions by Tumor Class', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

### 2.2  Feature Correlation Heatmap
Pearson correlation matrix of all 9 numeric features.

In [ ]:
# =============================================================================
# 2.2 Correlation Heatmap
# =============================================================================

corr = df[FEATURES].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

### 2.3  Box Plots per Class
For each feature, a box plot split by tumor label.  Highlights median,
quartiles, and outliers.

In [ ]:
# =============================================================================
# 2.3 Box Plots — Features by Class
# =============================================================================

fig, axes = plt.subplots(3, 3, figsize=(18, 12))

for ax, feat in zip(axes.flat, FEATURES):
    sns.boxplot(data=df, x='label', y=feat, ax=ax,
                palette='Set2', hue='label', legend=False)
    ax.set_title(feat, fontsize=12)
    ax.set_xlabel('')

fig.suptitle('Feature Box Plots by Tumor Class', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

### 2.4  Anatomical View Distribution
Stacked bar chart: how many images of each **view** (axial / coronal /
sagittal) exist per class.

In [ ]:
# =============================================================================
# 2.4 View Distribution by Class
# =============================================================================

view_counts = df.groupby(['label', 'view']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(8, 5))
view_counts.plot(kind='bar', stacked=True, ax=ax, colormap='tab10')
ax.set_ylabel('Count')
ax.set_title('Anatomical View Distribution per Class', fontsize=14)
ax.legend(title='View')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 2.5  Pair Plot (Selected Features)
Scatter matrix of the 4 most informative geometric + texture features,
colored by label.  Reveals pairwise class separation.

In [ ]:
# =============================================================================
# 2.5 Pair Plot — Selected Features
# =============================================================================

pair_features = ['area', 'solidity', 'eccentricity', 'contrast']

g = sns.pairplot(df[pair_features + ['label']],
                 hue='label', palette='Set1',
                 diag_kind='kde', plot_kws={'alpha': 0.5, 's': 15})
g.figure.suptitle('Pair Plot — Selected Features', y=1.02, fontsize=15)
plt.show()

## Feature Importance (Bonus)

**Goal:** Determine which extracted features contribute most to tumor classification.

We train a lightweight **Random Forest** classifier on the extracted features and rank relevance via built-in Mean Decrease in Impurity.

**This section includes three enhancements:**

1. **IQR-based outlier removal** — rows with any feature value beyond 1.5 × IQR are dropped before training, reducing noise for the Random Forest.
2. **Full classification report** — per-class precision, recall, and F1-score on the held-out test split.
3. **Confusion matrix heatmap** — visual breakdown of correct vs. mis-classified tumor types.


In [ ]:
# =============================================================================
# Step 1 — IQR-based outlier removal on training data
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

FEATURES = ['area', 'perimeter', 'solidity', 'eccentricity',
            'bbox_ratio', 'contrast', 'correlation', 'energy', 'homogeneity']

# ---- Split raw data ----
train_df = df[df['split'] == 'train'].copy()
test_df  = df[df['split'] == 'test'].copy()

def remove_iqr_outliers(data: pd.DataFrame, cols: list, factor: float = 1.5) -> pd.DataFrame:
    """Drop rows where any column value falls outside [Q1 - factor*IQR, Q3 + factor*IQR]."""
    mask = pd.Series(True, index=data.index)
    for col in cols:
        Q1, Q3 = data[col].quantile(0.25), data[col].quantile(0.75)
        IQR    = Q3 - Q1
        mask  &= data[col].between(Q1 - factor * IQR, Q3 + factor * IQR)
    return data[mask]

train_clean = remove_iqr_outliers(train_df, FEATURES)
dropped     = len(train_df) - len(train_clean)
pct_dropped = (dropped / len(train_df) * 100) if len(train_df) > 0 else 0
print(f"Outlier removal: {dropped} rows dropped ({pct_dropped:.1f}% of training data)")

# ---- Encode labels ----
le = LabelEncoder()
y_train = le.fit_transform(train_clean['label'])
X_train = train_clean[FEATURES]


### Train Random Forest & Feature Importances

In [ ]:
# =============================================================================
# Step 2 & 3 — Train Random Forest & Feature Importance bar chart
# =============================================================================
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

importances = rf.feature_importances_
sorted_idx  = importances.argsort()
sorted_feat = [FEATURES[i] for i in sorted_idx]
sorted_imp  = importances[sorted_idx]

fig, ax = plt.subplots(figsize=(9, 5))
colors  = plt.cm.viridis(np.linspace(0.25, 0.85, len(sorted_feat)))
bars    = ax.barh(sorted_feat, sorted_imp, color=colors)

for bar, val in zip(bars, sorted_imp):
    ax.text(val + 0.003, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_xlabel('Importance (MDI)')
ax.set_title('Feature Importance — Random Forest (after IQR cleaning)', fontsize=14)
plt.tight_layout()
plt.show()

print('\nFeature Importance Ranking:')
print('-' * 35)
for feat, imp in zip(reversed(sorted_feat), reversed(sorted_imp)):
    print(f'  {feat:15s}  {imp:.4f}')


### Evaluate on Test Split
We use the held-out test split to generate a full classification report and a confusion matrix heatmap.

In [ ]:
# =============================================================================
# Step 4 & 5 — Evaluate on test split: accuracy + classification report + confusion matrix
# =============================================================================
print(f'\nRandom Forest train accuracy: {rf.score(X_train, y_train):.4f}')

if len(test_df) > 0:
    y_test  = le.transform(test_df['label'])
    X_test  = test_df[FEATURES]
    y_pred  = rf.predict(X_test)

    print(f'Random Forest test  accuracy: {rf.score(X_test, y_test):.4f}')
    print('\nClassification Report (test set):')
    print('=' * 55)
    print(classification_report(y_test, y_pred, target_names=le.classes_))

    cm  = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(7, 5))
    im  = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.colorbar(im, ax=ax)

    classes = le.classes_
    tick_marks = np.arange(len(classes))
    ax.set_xticks(tick_marks); ax.set_xticklabels(classes, rotation=30, ha='right')
    ax.set_yticks(tick_marks); ax.set_yticklabels(classes)

    # Annotate each cell
    thresh = cm.max() / 2
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j],
                    ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black',
                    fontsize=13, fontweight='bold')

    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_ylabel('True Label',      fontsize=12)
    ax.set_title('Confusion Matrix — Random Forest (test set)', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No test split found — skipping evaluation metrics.')
